In [ ]:
from google.colab import files
file=files.upload()

Saving insurance.csv to insurance (1).csv


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data=pd.read_csv('insurance.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [ ]:
data.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


In [ ]:
data[data.duplicated(keep=False)]

,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data['sex'].value_counts()
display(data['sex'].value_counts())
display(data['smoker'].value_counts())
display(data['region'].value_counts())

,count
sex,
male,675
female,662


,count
smoker,
no,1063
yes,274


,count
region,
southeast,364
southwest,325
northwest,324
northeast,324


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
le_sex= LabelEncoder()
le_smoker=LabelEncoder()

In [ ]:
le_sex.fit(data['sex'].drop_duplicates())
le_smoker.fit(data['smoker'].drop_duplicates())

LabelEncoder()

In [ ]:
data['sex_enc']=le_sex.transform(data['sex'])
data['smoker_enc']=le_smoker.transform(data['smoker'])

In [ ]:
data.head()

,age,sex,bmi,children,smoker,region,charges,sex_enc,smoker_enc
0,19,female,27.900,0,yes,southwest,16884.92400,0,1
1,18,male,33.770,1,no,southeast,1725.55230,1,0
2,28,male,33.000,3,no,southeast,4449.46200,1,0
3,33,male,22.705,0,no,northwest,21984.47061,1,0
4,32,male,28.880,0,no,northwest,3866.85520,1,0


In [ ]:
ct = ColumnTransformer( [ ('ohe', OneHotEncoder(), ['region']) ],
remainder='passthrough' )
trans=ct.fit_transform(data)

In [ ]:
ins_data=pd.DataFrame(trans,columns=ct.get_feature_names_out())
list(ins_data.columns)
ins_data.head()

,ohe__region_northeast,ohe__region_northwest,ohe__region_southeast,ohe__region_southwest,remainder__age,remainder__sex,remainder__bmi,remainder__children,remainder__smoker,remainder__charges,remainder__sex_enc,remainder__smoker_enc
0,0.0,0.0,0.0,1.0,19,female,27.9,0,yes,16884.924,0,1
1,0.0,0.0,1.0,0.0,18,male,33.77,1,no,1725.5523,1,0
2,0.0,0.0,1.0,0.0,28,male,33.0,3,no,4449.462,1,0
3,0.0,1.0,0.0,0.0,33,male,22.705,0,no,21984.47061,1,0
4,0.0,1.0,0.0,0.0,32,male,28.88,0,no,3866.8552,1,0


In [ ]:
ins_data.columns = ['region_northeast',
'region_northwest',
'region_southeast',
'region_southwest',
'age',
'sex',
'bmi',
'children',
'smoker',
'charges',
'sex_enc',
'smoker_enc']

In [ ]:
ins_data=ins_data[['age',
'sex',
'sex_enc',
'bmi',
'children',
'smoker',
'smoker_enc',
'region_northeast',
'region_northwest',
'region_southeast',
'region_southwest',
'charges'
]]

In [ ]:
ins_data_t= ins_data[[ 'age',
'sex_enc',
'bmi',
'children',
'smoker_enc',
'region_northeast',
'region_northwest',
'region_southeast',
'region_southwest',
'charges'
]]
ins_data_t = ins_data_t.apply(pd.to_numeric)
ins_data_t.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               1338 non-null   int64  
 1   sex_enc           1338 non-null   int64  
 2   bmi               1338 non-null   float64
 3   children          1338 non-null   int64  
 4   smoker_enc        1338 non-null   int64  
 5   region_northeast  1338 non-null   float64
 6   region_northwest  1338 non-null   float64
 7   region_southeast  1338 non-null   float64
 8   region_southwest  1338 non-null   float64
 9   charges           1338 non-null   float64
dtypes: float64(6), int64(4)
memory usage: 104.7 KB


In [ ]:
from sklearn.model_selection import train_test_split
df_feat = ins_data_t [['age',
'sex_enc',
'bmi',
'children',
'smoker_enc',
'charges'
]]

In [ ]:
X= df_feat.iloc[:,0:-1]
y= df_feat.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=5,
test_size=0.3)

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [ ]:
#y = a + B*X
#a = model.intercept
#B = model.coef_
model.intercept_, model.coef_

(np.float64(-11974.904688367216),
 array([  260.23792394,  -316.20772806,   319.46943987,   598.28297551,
        24017.82960349]))

In [ ]:
y_pred = model.predict(X_test)
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
mse = mean_squared_error(y_pred, y_test)
sqrt_mse = np.sqrt(mse)
mae = mean_absolute_error(y_pred, y_test)

In [ ]:
print(f"MSE : {mse:.3f}, MSE_SQRT : {sqrt_mse:.3f}, MAE : {mae:.3f}")
r2 = model.score(X_test, y_test)
print(f"R2 score: {r2:.3f}")

MSE : 34679442.057, MSE_SQRT : 5888.925, MAE : 4087.666
R2 score: 0.756


In [ ]:
df_feat['charges'].min(), df_feat['charges'].max(),
df_feat['charges'].max()-df_feat['charges'].min()
df_feat.columns

Index(['age', 'sex_enc', 'bmi', 'children', 'smoker_enc', 'charges'], dtype='object')

In [ ]:
val = model.predict([[50,1, 45.9, 1, 0,]])
print('Predicted Insurance Charge =', val)

Predicted Insurance Charge = [15982.71404619]
